# 35. Voice & Realtime

**Tier:** Frontier
**Estimated time:** 40 minutes
**Prerequisites:** 13, 29
**Priority:** 🟢 Nice-to-have — valuable breadth and a growing market, but orthogonal to the core agent/eval stack; latency-budget thinking is the transferable part. *If skipped, revisit when:* a product needs voice in/out, or realtime UX (interruption handling) enters your roadmap.
**Source material:** Stanford Lecture 3 (inference optimization, latency); OpenAI's audio API documentation

## What You'll Learn
- The STT → LLM → TTS pipeline shape, and where each stage's latency comes from
- Building a per-stage latency budget for a target "feels natural" response time
- A real speech round-trip: generate audio from text, then transcribe it back, using the OpenAI audio API
- Streaming and interruption ("barge-in") handling — why a voice agent can't just wait for a full LLM response before speaking

## Why This Matters
Text chat tolerates a 2-second pause without feeling broken; voice conversation does not — humans expect a reply within a few hundred milliseconds or the interaction feels laggy and unnatural. Every concept from notebook 29 (streaming, timeouts) and notebook 13 (KV cache, why generation speed matters) resurfaces here with a much tighter budget, plus a genuinely new problem: what happens when the user starts talking while the agent is still speaking?


In [ ]:
import os, time, io

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

HAS_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
if HAS_OPENAI:
    from openai import OpenAI
    openai_client = OpenAI()
    print("OpenAI client ready (audio calls below will attempt live API use).")
else:
    openai_client = None
    print("No OPENAI_API_KEY — audio cells will explain instead of call.")


## The STT → LLM → TTS pipeline

A voice agent chains three stages, each with its own latency profile:

1. **STT (speech-to-text)** — audio in, transcript out. Streaming STT can start emitting partial transcripts before the user finishes talking.
2. **LLM** — transcript in, response text out. The same generation-speed concerns as notebook 11 (speculative decoding) and notebook 13 (KV cache) apply directly — a slower model here is a slower conversation.
3. **TTS (text-to-speech)** — response text in, audio out, ideally started on the FIRST few words rather than waiting for the whole response, so the user hears speech start almost immediately.

None of these stages is unique to voice — you already know STT-adjacent concepts (transcription) and TTS is "just" generation in a different modality. What's new is the END-TO-END LATENCY BUDGET: text chat tolerates seconds, voice tolerates a few hundred milliseconds before it feels broken.

## Building a latency budget

Given a target "feels natural" total latency, allocate it across the three stages, then check whether realistic per-stage latencies actually fit. This is the same budgeting discipline as notebook 32's per-feature token budgets, applied to time instead of tokens.

In [ ]:
TARGET_TOTAL_MS = 800   # a commonly cited threshold for "feels like a natural pause," not an interruption

STAGE_BUDGET_MS = {
    "stt_first_partial": 150,    # time to first partial transcript, not full utterance
    "llm_first_token": 300,      # time to first generated token (not full response)
    "tts_first_audio": 150,      # time to first audible audio chunk
    "network_overhead": 100,     # round-trips, buffering, jitter
}

total_budgeted = sum(STAGE_BUDGET_MS.values())
print(f"Target: {TARGET_TOTAL_MS}ms   Budgeted: {total_budgeted}ms   "
      f"{'OK — within budget' if total_budgeted <= TARGET_TOTAL_MS else 'OVER BUDGET — a stage must shrink'}")
for stage, ms in STAGE_BUDGET_MS.items():
    print(f"  {stage:20s} {ms:4d}ms  ({ms / TARGET_TOTAL_MS:.0%} of target)")


## The key trick: streaming, not waiting

Notice the budget above is for FIRST partial/token/audio, not the complete stage output — this is the same streaming principle from notebook 29, applied to every stage of the pipeline instead of just the LLM. Waiting for a complete transcript, then a complete LLM response, then generating complete audio would stack full-stage latencies sequentially and blow past any natural-feeling budget; streaming lets stages overlap — TTS can start on the first sentence while the LLM is still generating the second.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

# Illustrative full-stage latencies (NOT first-token/partial — the "if you don't stream" case).
FULL_STAGE_MS = {"stt": 1200, "llm": 2000, "tts": 900}

fig, ax = plt.subplots(figsize=(8, 3))
# Sequential (no streaming): stages stack end to end.
left = 0
for stage, ms in FULL_STAGE_MS.items():
    ax.barh("sequential\n(no streaming)", ms, left=left, label=f"{stage} ({ms}ms)")
    left += ms
# Streamed: each stage starts once the PREVIOUS stage's first chunk is available, so they overlap.
overlap_start = {"stt": 0, "llm": STAGE_BUDGET_MS["stt_first_partial"],
                 "tts": STAGE_BUDGET_MS["stt_first_partial"] + STAGE_BUDGET_MS["llm_first_token"]}
for stage, ms in FULL_STAGE_MS.items():
    ax.barh("streamed", ms, left=overlap_start[stage])

ax.set_xlabel("time (ms)")
ax.set_title("Streaming lets pipeline stages overlap instead of stacking")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()


*Sequential (wait-for-complete-stage) processing stacks each stage's FULL latency; streaming starts each downstream stage as soon as the first usable chunk from the upstream stage arrives, so the end-to-end wall-clock is much closer to the slowest single stage than the sum of all three.*

## A real round trip: text → speech → text

We generate real audio from text using OpenAI's TTS API, then transcribe that audio back to text using STT — a genuine round trip you can inspect, without needing a live microphone. Every call is wrapped so the notebook still runs clean if the API is unavailable (rate-limited, no credit, or no key at all).

In [ ]:
ORIGINAL_TEXT = "The KV cache stores past attention keys and values so decoding does not recompute them."

def text_to_speech(text, path="/tmp/nb35_speech.mp3"):
    if not HAS_OPENAI:
        return None, "[skipped: no OPENAI_API_KEY]"
    try:
        t0 = time.time()
        resp = openai_client.audio.speech.create(model="tts-1", voice="alloy", input=text)
        resp.stream_to_file(path)
        return path, f"generated in {time.time() - t0:.2f}s"
    except Exception as e:
        return None, f"[skipped: {type(e).__name__}: {str(e)[:150]}]"

def speech_to_text(path):
    if not HAS_OPENAI or path is None:
        return "[skipped: no audio to transcribe]"
    try:
        with open(path, "rb") as f:
            transcript = openai_client.audio.transcriptions.create(model="whisper-1", file=f)
        return transcript.text
    except Exception as e:
        return f"[skipped: {type(e).__name__}: {str(e)[:150]}]"

audio_path, tts_status = text_to_speech(ORIGINAL_TEXT)
print("TTS:", tts_status)
transcribed_text = speech_to_text(audio_path)
print("STT round-trip result:", transcribed_text)
print("\nOriginal: ", ORIGINAL_TEXT)


## Interruption ("barge-in") handling

Text chat has no equivalent of this: what happens when the user starts talking WHILE the agent's TTS audio is still playing? A voice agent needs a turn-taking state machine that can detect user speech mid-playback, immediately stop its own audio, and start listening — otherwise the agent talks over the user, which breaks the conversation far more than a slow response would.

In [ ]:
class VoiceTurnState:
    IDLE, LISTENING, THINKING, SPEAKING = "idle", "listening", "thinking", "speaking"

    def __init__(self):
        self.state = self.IDLE
        self.log = []

    def user_starts_speaking(self):
        if self.state == self.SPEAKING:
            self.log.append("BARGE-IN: user interrupted agent speech -> stopping TTS playback immediately")
        self.state = self.LISTENING
        self.log.append(f"state -> {self.state}")

    def user_stops_speaking(self):
        self.state = self.THINKING
        self.log.append(f"state -> {self.state}")

    def agent_starts_speaking(self):
        self.state = self.SPEAKING
        self.log.append(f"state -> {self.state}")

    def agent_finishes_speaking(self):
        self.state = self.IDLE
        self.log.append(f"state -> {self.state}")

turn = VoiceTurnState()
turn.user_starts_speaking()
turn.user_stops_speaking()
turn.agent_starts_speaking()
turn.user_starts_speaking()   # the user interrupts mid-response — a barge-in
for line in turn.log:
    print(line)


## Exercises

**Exercise 1 (Warm-up):** Change `TARGET_TOTAL_MS` to `400` (a stricter target) and re-run the budget check. Which stage(s) would need to shrink, and by how much, to hit that target?

**Exercise 2 (Apply):** Extend `VoiceTurnState` with a `agent_interrupted_count` counter that increments every time a barge-in occurs, and a method `summary()` that reports total turns and interruption rate.

**Exercise 3 (Extend):** Notebook 29 built a served FastAPI app with SSE streaming for text. Sketch how you'd adapt that same server to stream TTS AUDIO CHUNKS instead of text tokens — what changes about the SSE payload format, and what new failure mode (compared to text streaming) do you need to handle?


In [ ]:
# Exercise 1: Warm-up
# Task: Set TARGET_TOTAL_MS = 400 and determine which stage(s) must shrink to fit.
# Hint: reuse the STAGE_BUDGET_MS dict and the total_budgeted comparison from the cell above.

# YOUR CODE HERE


# Exercise 2: Apply
# Task: Add agent_interrupted_count and summary() to VoiceTurnState.
# Hint: increment the counter inside the existing "if self.state == self.SPEAKING" branch.

# YOUR CODE HERE


# Exercise 3: Extend
# Task: Sketch adapting notebook 29's /ask_stream SSE endpoint to stream TTS audio chunks.
# Hint: SSE payloads are text-based (base64-encode binary audio chunks); think about what
# happens if a barge-in occurs mid-stream and the client needs to tell the server to stop.

# YOUR CODE HERE


<details>
<summary>Click to reveal solutions</summary>

```python
# Exercise 1
TARGET_TOTAL_MS_STRICT = 400
overage = total_budgeted - TARGET_TOTAL_MS_STRICT
print(f"Over budget by {overage}ms — every stage would need to shrink roughly proportionally,")
print(f"or a stage would need to be cut entirely (e.g. skip network round-trip via on-device STT).")

# Exercise 2
class VoiceTurnStateV2(VoiceTurnState):
    def __init__(self):
        super().__init__()
        self.agent_interrupted_count = 0
        self.total_turns = 0

    def user_starts_speaking(self):
        if self.state == self.SPEAKING:
            self.agent_interrupted_count += 1
        self.total_turns += 1
        super().user_starts_speaking()

    def summary(self):
        rate = self.agent_interrupted_count / self.total_turns if self.total_turns else 0
        return {"total_turns": self.total_turns, "interruptions": self.agent_interrupted_count,
                "interruption_rate": rate}

# Exercise 3
# SSE payloads are text, so binary audio chunks need base64 encoding per event:
#   data: {"audio_chunk_b64": "<base64 mp3/pcm bytes>", "seq": 3}
# The NEW failure mode versus text streaming: a barge-in means the CLIENT needs to tell the
# SERVER to stop generating/sending further chunks mid-stream (not just stop playing locally),
# which text streaming never needed — add a cheap "stop" signal (e.g. a companion endpoint or
# a WebSocket control message) the server checks before emitting each next chunk.
```
</details>

## Key Takeaways
- Voice pipelines chain STT → LLM → TTS, and each stage's concepts are things you already know (transcription, generation speed from notebooks 11/13, streaming from notebook 29) — the new constraint is the END-TO-END latency budget.
- Budget for FIRST partial/token/audio per stage, not full-stage completion — streaming lets stages overlap instead of stacking latencies sequentially.
- A real text → speech → text round trip is a good way to sanity-check a voice pipeline without needing a live microphone.
- Barge-in (the user interrupting the agent mid-speech) has no text-chat equivalent — a voice agent needs an explicit turn-taking state machine to handle it, or it talks over its users.
- These are the same budgeting and streaming disciplines from notebooks 29 and 32, applied to a much tighter time constraint.

## What's Next
Notebook 36 covers data engineering for AI systems — the intersection where a Data Engineer's existing skills become this curriculum's most distinctive advantage.
